In [38]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')


# 1. Load the Dataset

In [39]:
df = pd.read_excel('../data/raw_data.xlsx',skiprows=1)
df.shape


(14531, 41)

# 2. Data Cleaning

In [40]:
# Clean up column names by removing leading/trailing whitespace
df.columns = df.columns.str.strip()


In [41]:
# Drop duplicate rows
initial_rows = len(df)
df.drop_duplicates(inplace=True)
print(f"Removed {initial_rows - len(df)} duplicate rows.")
print(df.shape)


Removed 0 duplicate rows.
(14531, 41)


### Drop columns that are empty

In [42]:
# Find columns that contain only missing values
dropped_columns = df.columns[df.isna().all()].tolist()

# Drop them
df.dropna(axis=1, how='all', inplace=True)

print(f"Dropped {len(dropped_columns)} columns.")
print("Dropped columns:")
print(dropped_columns)

print(f"New shape: {df.shape}")


Dropped 0 columns.
Dropped columns:
[]
New shape: (14531, 41)


### Rename columns

In [43]:
df.columns

Index(['Heure', 'Angle du vent(°)', 'Échelle vitesse vent()',
       'Vitesse vent(m/s)', 'Humidité ambiante(%RH)',
       'Température ambiante(℃)', 'Temp. (module PV)(℃)',
       'Irradiation journalière pente(Wh/㎡)',
       'Irradiation transitoire pente(W/㎡)', 'Radiation transitoire(W/㎡)',
       'Inverter1/Puissance DC totale (kW)',
       'Inverter1/Puissance active totale (kW)', 'pm10 μg/m³', 'pm2_5 μg/m³',
       'carbon_monoxide μg/m³', 'nitrogen_dioxide μg/m³',
       'sulphur_dioxide μg/m³', 'ozone μg/m³',
       'aerosol_optical_depth Dimensionless', 'dust μg/m³', 'uv_index Index',
       'uv_index_clear_sky Index', 'ammonia μg/m³', 'alder_pollen Grains/m³',
       'birch_pollen Grains/m³', 'grass_pollen Grains/m³',
       'mugwort_pollen Grains/m³', 'olive_pollen Grains/m³',
       'ragweed_pollen Grains/m³', 'rain (mm)', 'dew_point_2m (°C)',
       'precipitation (mm)', 'cloud_cover (%)', 'cloud_cover_low (%)',
       'cloud_cover_mid (%)', 'cloud_cover_high (%)', 'wind_g

In [44]:
column_mapping = {
    'Heure': 'Timestamp',
    'Angle du vent(°)': 'Wind_Dir_deg',
    'Vitesse vent(m/s)': 'Wind_Speed_ms',
    'Humidité ambiante(%RH)': 'Humidity_pct',
    'Température ambiante(℃)': 'Outdoor_Temp_C',
    'Temp. (module PV)(℃)': 'PV_Temperature',
    'Irradiation journalière pente(Wh/㎡)': 'Daily_Irradiation_Cumulated',
    'Irradiation transitoire pente(W/㎡)': 'Solar_Radiation_Wm2',
    'Inverter1/Puissance active totale (kW)': 'AC',
    'Inverter1/Puissance DC totale (kW)': 'DC',
    'Point de Rosée °C': 'Dew_Point_C',
    'rain (mm)': 'Rain_mm',
    'pm2_5 μg/m³':'PM25_ugm3',
    'pm10 μg/m³': 'PM10_ugm3',    
}

df.rename(columns=column_mapping, inplace=True)
print("Renamed key columns for clarity.")


Renamed key columns for clarity.


In [45]:
df.columns


Index(['Timestamp', 'Wind_Dir_deg', 'Échelle vitesse vent()', 'Wind_Speed_ms',
       'Humidity_pct', 'Outdoor_Temp_C', 'PV_Temperature',
       'Daily_Irradiation_Cumulated', 'Solar_Radiation_Wm2',
       'Radiation transitoire(W/㎡)', 'DC', 'AC', 'PM10_ugm3', 'PM25_ugm3',
       'carbon_monoxide μg/m³', 'nitrogen_dioxide μg/m³',
       'sulphur_dioxide μg/m³', 'ozone μg/m³',
       'aerosol_optical_depth Dimensionless', 'dust μg/m³', 'uv_index Index',
       'uv_index_clear_sky Index', 'ammonia μg/m³', 'alder_pollen Grains/m³',
       'birch_pollen Grains/m³', 'grass_pollen Grains/m³',
       'mugwort_pollen Grains/m³', 'olive_pollen Grains/m³',
       'ragweed_pollen Grains/m³', 'Rain_mm', 'dew_point_2m (°C)',
       'precipitation (mm)', 'cloud_cover (%)', 'cloud_cover_low (%)',
       'cloud_cover_mid (%)', 'cloud_cover_high (%)', 'wind_gusts_10m (km/h)',
       'wind_direction_100m (°)', 'wind_direction_10m (°)',
       'wind_speed_100m (km/h)', 'wind_speed_10m (km/h)'],
      dty

### Convert 'Hour' column to datetime objects

In [46]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')


### Remove rows before the meteostation started operating

In [47]:
cutoff = pd.Timestamp('2024-05-18 11:00:00')
df = df[df['Timestamp'] >= cutoff].reset_index(drop=True)
print(f"Removed rows before {cutoff}. New shape: {df.shape}")

Removed rows before 2024-05-18 11:00:00. New shape: (14520, 41)


### Replace '--' with NumPy's Not a Number (NaN)

In [48]:
# Replace placeholder characters ('--', etc.) with NumPy's Not a Number (NaN)
df.replace(['--', 'NA', 'NaN', 'nan'], np.nan, inplace=True)
print("Replaced placeholder text with NaN")


Replaced placeholder text with NaN


## Convert columns to numeric type

In [49]:
cols_to_convert = [
    'Wind_Angle', 'Wind_Speed',
    'Ambient_Humidity', 'Ambient_Temperature', 'PV_Temperature',
    'Transient_Irradiation',
    'Radiation transitoire(W/㎡)', 'DC',
    'pm10 μg/m³', 'pm2_5 μg/m³', 'uv_index Index',
    'rain (mm)', 'dew_point_2m (°C)',
    'precipitation (mm)', 'wind_gusts_10m (km/h)',
    'wind_direction_100m (°)', 'wind_direction_10m (°)',
    'wind_speed_100m (km/h)', 'wind_speed_10m (km/h)'
]

missing = [c for c in cols_to_convert if c not in df.columns]
if missing:
    print("Columns not found in df (check):", missing)

for col in cols_to_convert:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(',', '.', regex=False)
            .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("Converted all feature columns to numeric type.")


Columns not found in df (check): ['Wind_Angle', 'Wind_Speed', 'Ambient_Humidity', 'Ambient_Temperature', 'Transient_Irradiation', 'pm10 μg/m³', 'pm2_5 μg/m³', 'rain (mm)']
Converted all feature columns to numeric type.


In [50]:
### Convert 'DC'to W instead of KW

In [51]:
df['DC'] = df['DC'] * 1000
print("Converted DC from kW to W.")

Converted DC from kW to W.


### Drop unnecessary columns

In [52]:
# Drop columns that are not needed for the analysis
cols_to_drop = [
    'Daily_Irradiation_Cumulated',
    'AC',  # Puissance active (renamed to AC) - not needed
    'cloud_cover (%)',
    'cloud_cover_low (%)',
    'cloud_cover_mid (%)',
    'cloud_cover_high (%)',
    'carbon_monoxide μg/m³',
    'nitrogen_dioxide μg/m³',
    'sulphur_dioxide μg/m³',
    'ozone μg/m³',
    'ammonia μg/m³',
    'aerosol_optical_depth Dimensionless',
    'dust μg/m³',
    'uv_index_clear_sky Index',
    'alder_pollen Grains/m³',
    'birch_pollen Grains/m³',
    'grass_pollen Grains/m³',
    'mugwort_pollen Grains/m³',
    'olive_pollen Grains/m³',
    'ragweed_pollen Grains/m³',
    'Échelle vitesse vent()',
    'Précipitation Totale mm',
    'wind_gusts_10m (km/h)',
    'wind_direction_100m (°)',
    'wind_direction_10m (°)', 
    'wind_speed_100m (km/h)',
    'wind_speed_10m (km/h)',
    'precipitation (mm)',
    'Radiation transitoire(W/㎡)'
]

existing_cols_to_drop = [c for c in cols_to_drop if c in df.columns]
missing_cols_to_drop = [c for c in cols_to_drop if c not in df.columns]

df.drop(columns=existing_cols_to_drop, inplace=True)

print(f"Dropped {len(existing_cols_to_drop)} columns:")
print(existing_cols_to_drop)
if missing_cols_to_drop:
    print("Not found in df (already absent):", missing_cols_to_drop)
print(f"New shape: {df.shape}")


Dropped 28 columns:
['Daily_Irradiation_Cumulated', 'AC', 'cloud_cover (%)', 'cloud_cover_low (%)', 'cloud_cover_mid (%)', 'cloud_cover_high (%)', 'carbon_monoxide μg/m³', 'nitrogen_dioxide μg/m³', 'sulphur_dioxide μg/m³', 'ozone μg/m³', 'ammonia μg/m³', 'aerosol_optical_depth Dimensionless', 'dust μg/m³', 'uv_index_clear_sky Index', 'alder_pollen Grains/m³', 'birch_pollen Grains/m³', 'grass_pollen Grains/m³', 'mugwort_pollen Grains/m³', 'olive_pollen Grains/m³', 'ragweed_pollen Grains/m³', 'Échelle vitesse vent()', 'wind_gusts_10m (km/h)', 'wind_direction_100m (°)', 'wind_direction_10m (°)', 'wind_speed_100m (km/h)', 'wind_speed_10m (km/h)', 'precipitation (mm)', 'Radiation transitoire(W/㎡)']
Not found in df (already absent): ['Précipitation Totale mm']
New shape: (14520, 13)


In [53]:
df.columns

Index(['Timestamp', 'Wind_Dir_deg', 'Wind_Speed_ms', 'Humidity_pct',
       'Outdoor_Temp_C', 'PV_Temperature', 'Solar_Radiation_Wm2', 'DC',
       'PM10_ugm3', 'PM25_ugm3', 'uv_index Index', 'Rain_mm',
       'dew_point_2m (°C)'],
      dtype='str')

In [54]:
df.shape

(14520, 13)

## Save cleaned data

In [55]:
df_cleaned = df
df_cleaned.to_csv("../data/cleaned_data.csv", index=False)
print(f"Saved cleaned data. Shape: {df_cleaned.shape}")


Saved cleaned data. Shape: (14520, 13)
